### Installing Libraries

In [ ]:
# Installing the HuggingFace Transformers Diffusers and dataset libraries
!pip install -q --upgrade transformers==4.56.2 diffusers==0.32.2 datasets==3.6.0

### Connecting to Tesla T4 GPU

In [ ]:
# GPU Information
gpu_info = !nvidia-smi
gpu_info

In [ ]:
# Check the T4 status
gpu_info = '\n'.join(gpu_info)
if 'Tesla T4' in gpu_info:
    print("Success - Connected to a T4")
else:
    print("NOT CONNECTED TO A T4")

### Connecting to HuggingFace

In [ ]:
import getpass
from huggingface_hub import login

hf_token = getpass.getpass("Enter HuggingFace Token: ")
login(token=hf_token, add_to_git_credential=True)

### Diffusers

- It Generate images by gradually denoising random noise
- Starts with random noise → iteratively refines → ends with a clear image

In [ ]:
import IPython as ipy
from IPython.display import display
from diffusers import AutoPipelineForText2Image
import torch

- Loads the SDXL-Turbo text-to-image model in half precision.
- Initializes a pipeline that can generate images from text prompts.
- Moves the model to the GPU for fast inference.

-----------

- variant is version of the weights.
- torch_dtype make sure what kind of numbers to use for model weights and computations.

#### stabilityai/sdxl-turbo

In [ ]:
pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo",  
    torch_dtype=torch.float16, 
    variant="fp16")
pipe.to("cuda")

prompt = "A class of data scientists learning AI engineering in a vibrant high-energy pop-art style"

image = pipe(prompt=prompt, num_inference_steps=4).images[0]
display(image)

In [ ]:
# Restart the kernel
import IPython as ipy
ipy.Application.instance().kernel.do_shutdown(True)

#### stabilityai/stable-diffusion-xl-base-1.0

In [ ]:
from IPython.display import display
from diffusers import DiffusionPipeline
import torch

In [ ]:
pipe = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0", 
    use_safetensors=True, 
    torch_dtype=torch.float16, 
    variant="fp16")
pipe.to("cuda")

In [ ]:
prompt = "A class of data scientists learning AI engineering in a vibrant high-energy pop-art style"
image = pipe(prompt=prompt, num_inference_steps=10).images[0]

display(image)

In [ ]:
# Restart the kernel
ipy.Application.instance().kernel.do_shutdown(True)

#### stabilityai/stable-diffusion-xl-base-1.0 and stabilityai/stable-diffusion-xl-refiner-1.0

- This will run the first 80% of the steps on the base model.
- The last 20% of the steps on the refiner model, which will take the noisy image from the base model and refine it to a higher quality image.

In [ ]:
from diffusers import DiffusionPipeline
import torch

base = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0", 
    torch_dtype=torch.float16, 
    variant="fp16", 
    use_safetensors=True)
base.to("cuda")

refiner = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-refiner-1.0", 
    text_encoder_2=base.text_encoder_2, 
    vae=base.vae, 
    torch_dtype=torch.float16, 
    use_safetensors=True, 
    variant="fp16",)
refiner.to("cuda")

# Define how many steps and what % of steps to be run on each experts (80/20) here
n_steps = 40
high_noise_frac = 0.8

prompt = "A class of data scientists learning AI engineering in a vibrant high-energy pop-art style"

# This outputs a latent image, which is a lower-dimensional representation of the image that captures its essential features.
image = base(
    prompt=prompt,
    num_inference_steps=n_steps,
    denoising_end=high_noise_frac,
    output_type="latent",
).images

#  The refiner can then take this latent image and further refine it to produce a higher-quality output.
image = refiner(
    prompt=prompt,
    num_inference_steps=n_steps,
    denoising_start=high_noise_frac,
    image=image,
).images[0]

display(image)

#### microsoft/speecht5_tts

In [ ]:
from transformers import pipeline
from datasets import load_dataset
import torch
from IPython.display import Audio

In [ ]:
# This creates a text-to-speech pipeline using the "microsoft/speecht5_tts" model
synthesizer = pipeline("text-to-speech", "microsoft/speecht5_tts", device='cuda')
# The embeddings capture the characteristics of different speakers, allowing the model to synthesize speech that mimics those characteristics.
embeddings_dataset = load_dataset("matthijs/cmu-arctic-xvectors", split="validation", trust_remote_code=True)
# Load the speaker embedding for a specific speaker and add a batch dimension to it.
speaker_embedding = torch.tensor(embeddings_dataset[7259]["xvector"]).unsqueeze(0)
speech = synthesizer("Hi to an artificial intelligence engineer, on the way to mastery!", forward_params={"speaker_embeddings": speaker_embedding})

In [ ]:
# 
Audio(speech["audio"], rate=speech["sampling_rate"],)